In [58]:
import os
import glob
import pandas as pd
from functools import reduce

from chain import Chain

In [59]:
csv_folder = "cris_data/"
output_file = "collisions_db.csv"

In [60]:
# Function to clean column names
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [col.strip().replace(" ", "_") for col in df.columns]
    return df

collision_etl_chain = (Chain()
    .map(lambda file: pd.read_csv(file, skiprows=11)) # load files
    .pipe(lambda dfs: pd.concat(dfs, ignore_index=True)) # concatenate DataFrames
    .pipe(clean_column_names) # clean column names
)

def DataPreprocessor(csv_folder: str, function_chain: Chain):
    def fetch_csv_files(csv_folder):
        return glob.glob(os.path.join(csv_folder, "*.csv"))
    
    csv_files = fetch_csv_files(csv_folder)
    combined_df = function_chain(csv_files)
    return combined_df

combined_df = DataPreprocessor(csv_folder, collision_etl_chain)

In [61]:
# --------------------------
# Extract first hour from range (e.g., "11:00 - 11:59")
# --------------------------
def extract_first_hour(hour_str: str) -> str:
    if pd.isna(hour_str):
        return pd.NA
    try:
        return str(hour_str).split(" - ")[0].strip()
    except:
        return pd.NA

# --------------------------
# Convert 24-hour string "HH:MM" to 12-hour integer
# --------------------------
def hour_24_to_12(hour_str: str) -> str:
    if pd.isna(hour_str):
        return pd.NA
    try:
        hour = int(hour_str.split(":")[0])
    except:
        return pd.NA

    if hour == 0:
        return "12 AM"
    elif hour < 12:
        return f"{hour} AM"
    elif hour == 12:
        return "12 PM"
    else:
        return f"{hour-12} PM"
    
def series_pipeline(*funcs):
    """
    Returns a function that applies a series of transformations to a Series.
    Each function should take a Series and return a Series.
    """
    def apply_pipeline(series: pd.Series) -> pd.Series:
        return reduce(lambda s, f: f(s), funcs, series)
    return apply_pipeline

# Create the pipeline: first extract first hour, then convert to 12-hour format
hour_pipeline = series_pipeline(
    lambda s: s.apply(extract_first_hour),
    lambda s: s.apply(hour_24_to_12)
)

# Apply the series pipeline and assign to the column
combined_df["Hour_of_Day"] = hour_pipeline(combined_df["Hour_of_Day"])
combined_df

,Crash_ID,At_Intersection_Flag,Crash_Date,Crash_Severity,Crash_Time,Crash_Year,Day_of_Week,Hour_of_Day,Intersecting_Street_Name,Latitude,...,Street_Name,Person_Death_Count,Person_Injury_Severity,Person_Non-Suspected_Serious_Injury_Count,Person_Not_Injured_Count,Person_Possible_Injury_Count,Person_Suspected_Serious_Injury_Count,Person_Total_Injury_Count,Person_Type,Person_Unknown_Injury_Count
0,19079854,True,2022-08-26,N - NOT INJURED,830,2022,FRIDAY,8 AM,EDINBURG DR,31.91536161,...,FM2529,0,99 - UNKNOWN,0,0,0,0,0,1 - DRIVER,1
1,19079854,True,2022-08-26,N - NOT INJURED,830,2022,FRIDAY,8 AM,EDINBURG DR,31.91536161,...,FM2529,0,N - NOT INJURED,0,1,0,0,0,1 - DRIVER,0
2,19079854,True,2022-08-26,N - NOT INJURED,830,2022,FRIDAY,8 AM,EDINBURG DR,31.91536161,...,FM2529,0,N - NOT INJURED,0,1,0,0,0,2 - PASSENGER/OCCUPANT,0
3,19080106,False,2022-08-26,N - NOT INJURED,1122,2022,FRIDAY,11 AM,NaN,31.91288964,...,SUN VALLEY DR,0,N - NOT INJURED,0,1,0,0,0,1 - DRIVER,0
4,19080106,False,2022-08-26,N - NOT INJURED,1122,2022,FRIDAY,11 AM,NaN,31.91288964,...,SUN VALLEY DR,0,N - NOT INJURED,0,1,0,0,0,2 - PASSENGER/OCCUPANT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163971,21300235,False,2026-02-27,N - NOT INJURED,1359,2026,FRIDAY,1 PM,NaN,31.76948396,...,US0062,0,N - NOT INJURED,0,1,0,0,0,2 - PASSENGER/OCCUPANT,0
163972,21299454,True,2026-02-28,N - NOT INJURED,950,2026,SATURDAY,9 AM,ELM ST,31.78464484,...,N PIEDRAS ST,0,99 - UNKNOWN,0,0,0,0,0,1 - DRIVER,1
163973,21299454,True,2026-02-28,N - NOT INJURED,950,2026,SATURDAY,9 AM,ELM ST,31.78464484,...,N PIEDRAS ST,0,N - NOT INJURED,0,1,0,0,0,1 - DRIVER,0
163974,21299993,False,2026-02-28,N - NOT INJURED,1530,2026,SATURDAY,3 PM,NaN,31.7780292,...,IH0010,0,99 - UNKNOWN,0,0,0,0,0,1 - DRIVER,1


In [62]:
def person_ethnicity_formatting(ethnicity_str: str) -> str:
    try:
        return ethnicity_str.split(" - ")[1].strip()
    except:
        return pd.NA

def person_gender_formatting(person_gender_str: str) -> str:
    try:
        return person_gender_str.split(" - ")[1].strip()
    except:
        return pd.NA

def person_type_formatting(person_type_str: str) -> str:
    try:
        return person_type_str.split(" - ")[1].strip()
    except:
        return pd.NA
    
# New mapping for Person_Type consolidation
PERSON_TYPE_MAP = {
    'DRIVER OF MOTORCYCLE TYPE VEHICLE': 'Motorcyclist',
    'PASSENGER/OCCUPANT ON MOTORCYCLE TYPE VEHICLE': 'Motorcyclist',
    'DRIVER': 'Motorist',
    'PASSENGER/OCCUPANT': 'Motorist',
    'PEDESTRIAN': 'Pedestrian',
    'PEDALCYCLIST': 'Cyclist',
    'OTHER (EXPLAIN IN NARRATIVE)': 'Other',
    'UNKNOWN': 'Unknown'
}

def person_type_consolidation(person_type_str: str) -> str:
    try:
        formatted = person_type_formatting(person_type_str)
        return PERSON_TYPE_MAP.get(formatted, 'Other')  # default to 'Other' if missing
    except:
        return pd.NA

pipeline_dict = {
    "Person_Ethnicity": series_pipeline(lambda s: s.apply(person_ethnicity_formatting)),
    "Person_Gender": series_pipeline(lambda s: s.apply(person_gender_formatting)),
    "Person_Type": series_pipeline(lambda s: s.apply(person_type_consolidation))
}

for column, pipeline in pipeline_dict.items():
    if column in combined_df.columns:
        combined_df[column] = pipeline(combined_df[column])

combined_df

,Crash_ID,At_Intersection_Flag,Crash_Date,Crash_Severity,Crash_Time,Crash_Year,Day_of_Week,Hour_of_Day,Intersecting_Street_Name,Latitude,...,Street_Name,Person_Death_Count,Person_Injury_Severity,Person_Non-Suspected_Serious_Injury_Count,Person_Not_Injured_Count,Person_Possible_Injury_Count,Person_Suspected_Serious_Injury_Count,Person_Total_Injury_Count,Person_Type,Person_Unknown_Injury_Count
0,19079854,True,2022-08-26,N - NOT INJURED,830,2022,FRIDAY,8 AM,EDINBURG DR,31.91536161,...,FM2529,0,99 - UNKNOWN,0,0,0,0,0,Motorist,1
1,19079854,True,2022-08-26,N - NOT INJURED,830,2022,FRIDAY,8 AM,EDINBURG DR,31.91536161,...,FM2529,0,N - NOT INJURED,0,1,0,0,0,Motorist,0
2,19079854,True,2022-08-26,N - NOT INJURED,830,2022,FRIDAY,8 AM,EDINBURG DR,31.91536161,...,FM2529,0,N - NOT INJURED,0,1,0,0,0,Motorist,0
3,19080106,False,2022-08-26,N - NOT INJURED,1122,2022,FRIDAY,11 AM,NaN,31.91288964,...,SUN VALLEY DR,0,N - NOT INJURED,0,1,0,0,0,Motorist,0
4,19080106,False,2022-08-26,N - NOT INJURED,1122,2022,FRIDAY,11 AM,NaN,31.91288964,...,SUN VALLEY DR,0,N - NOT INJURED,0,1,0,0,0,Motorist,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163971,21300235,False,2026-02-27,N - NOT INJURED,1359,2026,FRIDAY,1 PM,NaN,31.76948396,...,US0062,0,N - NOT INJURED,0,1,0,0,0,Motorist,0
163972,21299454,True,2026-02-28,N - NOT INJURED,950,2026,SATURDAY,9 AM,ELM ST,31.78464484,...,N PIEDRAS ST,0,99 - UNKNOWN,0,0,0,0,0,Motorist,1
163973,21299454,True,2026-02-28,N - NOT INJURED,950,2026,SATURDAY,9 AM,ELM ST,31.78464484,...,N PIEDRAS ST,0,N - NOT INJURED,0,1,0,0,0,Motorist,0
163974,21299993,False,2026-02-28,N - NOT INJURED,1530,2026,SATURDAY,3 PM,NaN,31.7780292,...,IH0010,0,99 - UNKNOWN,0,0,0,0,0,Motorist,1


In [63]:
severity_values = {
    'K - FATAL INJURY': 13749500,
    'A - SUSPECTED SERIOUS INJURY': 1432400,
    'B - SUSPECTED MINOR INJURY': 303200,
    'C - POSSIBLE INJURY': 151600,
    'N - NOT INJURED': 0,
    '99 - UNKNOWN': 0
}

combined_df["KABCO_USD_Value"] = combined_df["Crash_Severity"].map(severity_values).fillna(0)

In [64]:
combined_df[["Latitude", "Longitude"]] = combined_df[["Latitude", "Longitude"]].apply(
    pd.to_numeric, errors="coerce"
)

In [ ]:
filters_chain = (Chain()
    .pipe(lambda df: df.dropna(subset=["On_System_Flag"])) # drop rows where 'On System Flag' is NaN
    .pipe(lambda df: df.query("`On_System_Flag` == 'No'")) # filter rows where 'On System Flag' is 'No'
    .pipe(lambda df: df.query("`At_Intersection_Flag` == True")) # filter rows where 'At Intersection Flag' is 'No'
    .pipe(lambda df: df.query("`Person_Type` in ['Pedestrian', 'Cyclist']"))  # keep Motorists, Pedestrians, Cyclists
)

combined_df = filters_chain(combined_df)

combined_df.drop(columns=["On_System_Flag", 'Person_Non-Suspected_Serious_Injury_Count', 'Person_Not_Injured_Count',
       'Person_Possible_Injury_Count', 'Person_Suspected_Serious_Injury_Count',
       'Person_Total_Injury_Count', 'Person_Unknown_Injury_Count'], inplace=True)

combined_df.to_csv("collisions_db.csv", index=False)
combined_df

,Crash_ID,At_Intersection_Flag,Crash_Date,Crash_Severity,Crash_Time,Crash_Year,Day_of_Week,Hour_of_Day,Intersecting_Street_Name,Latitude,Longitude,Speed_Limit,Street_Name,Person_Death_Count,Person_Injury_Severity,Person_Type,KABCO_USD_Value
87,19086778,True,2022-08-26,B - SUSPECTED MINOR INJURY,753,2022,FRIDAY,7 AM,NaN,NaN,NaN,25,E HUNTER DR,0,B - SUSPECTED MINOR INJURY,Pedestrian,303200
116,19093623,True,2022-08-26,A - SUSPECTED SERIOUS INJURY,2324,2022,FRIDAY,11 PM,THORN AVE,31.850745,-106.58087,30,MACE ST,0,A - SUSPECTED SERIOUS INJURY,Pedestrian,1432400
1863,19114021,True,2022-09-12,N - NOT INJURED,727,2022,MONDAY,7 AM,WEXFORD DR,31.788905,-106.33899,35,EDGEMERE BLVD,0,N - NOT INJURED,Pedestrian,0
2529,19119237,True,2022-09-17,B - SUSPECTED MINOR INJURY,652,2022,SATURDAY,6 AM,RADFORD ST,31.788715,-106.43743,30,TROWBRIDGE DR,0,B - SUSPECTED MINOR INJURY,Pedestrian,303200
2888,19122648,True,2022-09-20,B - SUSPECTED MINOR INJURY,614,2022,TUESDAY,6 AM,HONOUR POINT PL,31.769235,-106.22883,35,MONTWOOD,0,B - SUSPECTED MINOR INJURY,Cyclist,303200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161626,21263664,True,2026-02-05,B - SUSPECTED MINOR INJURY,804,2026,THURSDAY,8 AM,NaN,31.797215,-106.24841,35,R C POE RD,0,B - SUSPECTED MINOR INJURY,Pedestrian,303200
161670,21265847,True,2026-02-05,A - SUSPECTED SERIOUS INJURY,1932,2026,THURSDAY,7 PM,AMECA ST,31.743795,-106.35847,30,ADOBE DR,0,A - SUSPECTED SERIOUS INJURY,Pedestrian,1432400
161873,21276571,True,2026-02-06,B - SUSPECTED MINOR INJURY,1640,2026,FRIDAY,4 PM,EDGEMERE BLVD,31.792165,-106.27636,35,NOLAN RICHARDSON DR,0,B - SUSPECTED MINOR INJURY,Pedestrian,303200
163730,21293830,True,2026-02-24,B - SUSPECTED MINOR INJURY,718,2026,TUESDAY,7 AM,STONE GATE LN,31.727995,-106.27943,35,WHIRLAWAY DR,0,B - SUSPECTED MINOR INJURY,Pedestrian,303200
